# Wages, Inflation, and Purchasing Power in Armenia (2019–2024)

This notebook reproduces the calculations and visualizations used in the project report.

**Research question:** Did growth in Armenia's average nominal wage between 2019 and 2024 translate into higher purchasing power after accounting for consumer-price inflation?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA = BASE / "data"
VIZ = BASE / "visualizations"

wages = pd.read_csv(DATA / "wages.csv")
cpi = pd.read_csv(DATA / "cpi.csv")
prices = pd.read_csv(DATA / "consumer_prices.csv")

wages, cpi, prices

## 1. Wage and inflation indicators

In [ ]:
df = wages.merge(cpi, on="year")
df["nominal_wage_growth_percent"] = df["average_monthly_nominal_wage_amd"].pct_change() * 100
df["wage_index_2019_100"] = df["average_monthly_nominal_wage_amd"] / df.loc[df.year==2019, "average_monthly_nominal_wage_amd"].iloc[0] * 100

price_index = [100.0]
for inflation in df.loc[df.year > 2019, "annual_average_inflation_percent"]:
    price_index.append(price_index[-1] * (1 + inflation/100))
df["cpi_index_2019_100"] = price_index
df["real_wage_index_2019_100"] = df["wage_index_2019_100"] / df["cpi_index_2019_100"] * 100
df.round(2)

In [ ]:
plt.figure(figsize=(9,5.5))
plt.plot(df.year, df.wage_index_2019_100, marker="o", label="Average nominal wage")
plt.plot(df.year, df.cpi_index_2019_100, marker="o", label="Consumer price level")
plt.axhline(100, linewidth=.8)
plt.title("Average Wage vs. Consumer Price Level in Armenia")
plt.xlabel("Year"); plt.ylabel("Index (2019 = 100)")
plt.legend(); plt.tight_layout()
plt.show()

## 2. Annual wage growth versus inflation

In [ ]:
growth = df.dropna(subset=["nominal_wage_growth_percent"])
growth[["year","nominal_wage_growth_percent","annual_average_inflation_percent"]].round(2)

## 3. Real wage index

In [ ]:
df[["year","real_wage_index_2019_100"]].round(2)

## 4. Retail-price affordability case study

In [ ]:
w2019 = df.loc[df.year==2019, "average_monthly_nominal_wage_amd"].iloc[0]
w2024 = df.loc[df.year==2024, "average_monthly_nominal_wage_amd"].iloc[0]
wage_change = (w2024/w2019 - 1)*100

prices["price_change_percent"] = (prices.price_2024/prices.price_2019 - 1)*100
prices["units_affordable_2019"] = w2019/prices.price_2019
prices["units_affordable_2024"] = w2024/prices.price_2024
prices["affordability_change_percent"] = (prices.units_affordable_2024/prices.units_affordable_2019 - 1)*100
prices.round(2)

## Interpretation

The project separates **nominal wage growth** from **real purchasing power**. With 2019 set to 100, the wage index rises faster than the constructed consumer-price index. The rice example provides a concrete affordability check: its price rose more slowly than the average wage, so the quantity purchasable with one average monthly wage increased.

This is a descriptive analysis, not a causal model. The average wage is not the same as household disposable income, and one retail item cannot represent the entire consumption basket.

## 5. A clearer retail-price comparison

Instead of showing an abstract percentage called “affordability change,” the next charts answer two concrete questions:

1. **Did the average wage or the rice price grow faster?**
2. **How many kilograms of rice could one average monthly wage buy in 2019 versus 2024?**

In [ ]:
rice_change = prices.loc[0, "price_change_percent"]
y_wage = [100, 100 * (1 + wage_change / 100)]
y_rice = [100, 100 * (1 + rice_change / 100)]

plt.figure(figsize=(8.5, 5.5))
plt.plot([2019, 2024], y_wage, marker="o", linewidth=2.5, label="Average wage")
plt.plot([2019, 2024], y_rice, marker="o", linewidth=2.5, label="Rice price")
plt.xticks([2019, 2024])
plt.ylabel("Index (2019 = 100)")
plt.title("Wages Rose Faster Than the Price of Rice")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
q2019 = prices.loc[0, "units_affordable_2019"]
q2024 = prices.loc[0, "units_affordable_2024"]

plt.figure(figsize=(7.5, 5.5))
bars = plt.bar(["2019", "2024"], [q2019, q2024], width=0.58)
plt.ylabel("Kilograms of rice")
plt.title("How Much Rice Could One Average Monthly Wage Buy?")
for bar, value in zip(bars, [q2019, q2024]):
    plt.text(bar.get_x() + bar.get_width()/2, value + 4, f"{value:.0f} kg", ha="center", fontweight="bold")
plt.ylim(0, max(q2019, q2024) * 1.18)
plt.tight_layout()
plt.show()